# 📈 Bitcoin ETL Project (Local Version)

In this project we will extract cryptocurrency market data from the CoinGecko API using the `pycoingecko` library and transforms it into a pandas DataFrame for further analysis as a pre cursor to when we use Amazon AWS!


In [7]:
https://docs.coingecko.com/reference/setting-up-your-api-key

SyntaxError: invalid syntax (1616845149.py, line 1)

In [8]:
# ✅ Install dependencies
!pip install pycoingecko pandas


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [9]:
# 📦 Import packages
from pycoingecko import CoinGeckoAPI
from datetime import datetime

This calls the get_coins_markets method from the CoinGecko API.
It fetches market data (like price, market cap, volume) for the top 50 coins.
vs_currency='usd' means all prices will be shown in US dollars.
per_page=50 means you’re asking for 50 results on one page.

API = Application Programming Interface
It’s a way for software programs to talk to each other. Like a waiter..
2 ways to connect:
    
API KEY: Basially a password. You get it from the API provider (like OpenWeather, YouTube, etc.)
Open Authorization 2( OAUTH2): Used to get access tokens(temporay badge access)
The API gives you a Client ID (public) and Client Secret (private)
You use these to request permission to access data on behalf of a user

API Key = Simple, direct access
Client ID + Secret = Used to get tokens (for login-style access)
Bearer Token = The actual token you use after being authorized

CoinGecko’s public API, no client_id, client_secret, or API key is required. It’s a free, public, unauthenticated API, which is why this line works out of the box

When you connect to most APIs, the service needs to know who you are and if you're allowed to access the data. You prove this using:

In [11]:
# 🔄 Extract Data from CoinGecko
cg = CoinGeckoAPI() #This creates a CoinGeckoAPI object from the pycoingecko library.

# Get top 50 cryptocurrencies by market cap in USD
raw_data = cg.get_coins_markets(vs_currency='usd', per_page=50)

raw_data


[{'id': 'bitcoin',
  'symbol': 'btc',
  'name': 'Bitcoin',
  'image': 'https://coin-images.coingecko.com/coins/images/1/large/bitcoin.png?1696501400',
  'current_price': 106059,
  'market_cap': 2108637548979,
  'market_cap_rank': 1,
  'fully_diluted_valuation': 2108637548979,
  'total_volume': 25488537485,
  'high_24h': 107216,
  'low_24h': 105402,
  'price_change_24h': -1113.7164334055706,
  'price_change_percentage_24h': -1.03918,
  'market_cap_change_24h': -22216837249.01831,
  'market_cap_change_percentage_24h': -1.04263,
  'circulating_supply': 19886246.0,
  'total_supply': 19886246.0,
  'max_supply': 21000000.0,
  'ath': 111814,
  'ath_change_percentage': -5.18374,
  'ath_date': '2025-05-22T18:41:28.492Z',
  'atl': 67.81,
  'atl_change_percentage': 156247.99235,
  'atl_date': '2013-07-06T00:00:00.000Z',
  'roi': None,
  'last_updated': '2025-07-02T03:24:50.749Z'},
 {'id': 'ethereum',
  'symbol': 'eth',
  'name': 'Ethereum',
  'image': 'https://coin-images.coingecko.com/coins/imag

📦 Step 1: Understand the Raw JSON (raw_data)

When we call the CoinGecko API , it returns a LIST of dictionaries.
Each dictionary represents one coin and contains a lot of details like name, symbol, price, etc.

In [12]:
import sys
print(sys.version)

3.13.3 (main, Apr  8 2025, 13:54:08) [Clang 16.0.0 (clang-1600.0.26.6)]


In [13]:
# Preview raw JSON
raw_data[:2]  # Display first 2 entries, aka all the columns but 2 rows

[{'id': 'bitcoin',
  'symbol': 'btc',
  'name': 'Bitcoin',
  'image': 'https://coin-images.coingecko.com/coins/images/1/large/bitcoin.png?1696501400',
  'current_price': 106059,
  'market_cap': 2108637548979,
  'market_cap_rank': 1,
  'fully_diluted_valuation': 2108637548979,
  'total_volume': 25488537485,
  'high_24h': 107216,
  'low_24h': 105402,
  'price_change_24h': -1113.7164334055706,
  'price_change_percentage_24h': -1.03918,
  'market_cap_change_24h': -22216837249.01831,
  'market_cap_change_percentage_24h': -1.04263,
  'circulating_supply': 19886246.0,
  'total_supply': 19886246.0,
  'max_supply': 21000000.0,
  'ath': 111814,
  'ath_change_percentage': -5.18374,
  'ath_date': '2025-05-22T18:41:28.492Z',
  'atl': 67.81,
  'atl_change_percentage': 156247.99235,
  'atl_date': '2013-07-06T00:00:00.000Z',
  'roi': None,
  'last_updated': '2025-07-02T03:24:50.749Z'},
 {'id': 'ethereum',
  'symbol': 'eth',
  'name': 'Ethereum',
  'image': 'https://coin-images.coingecko.com/coins/imag

In [ ]:
#let us first try to grab some data
raw_data[0]['id']

In [26]:
#remember
dic = {'apples':2, "pears":3, "grapes":8}
dic['apples']

2

In [31]:
raw_data[0]['symbol']

'btc'

In [30]:
raw_data[1]['roi']['percentage'] 

2933.487679457216

In [ ]:
we have the raw data and lets say we are a finance company who needs to have organized groupings of data or DATASETS this is how you would
do it. 
Logically we have metadata and market data

🧱 Step 2: Build a Clean List of Metadata

We only want certain descriptive fields about each coin (not the changing prices or volumes). So we loop through the list and extract the metadata fields we care about:

In [14]:
metadata_list = []

for coin in raw_data:
    coin_id = coin['id']
    coin_symbol = coin['symbol']
    coin_name = coin['name']
    coin_image = coin['image']
    coin_rank = coin['market_cap_rank']
    coin_max_supply = coin['max_supply']
#We group the selected fields into a dictionary so we can treat each coin’s data like a row:
    coin_metadata = {
        'coin_id': coin_id,
        'symbol': coin_symbol,
        'name': coin_name,
        'image_url': coin_image,
        'market_cap_rank': coin_rank,
        'max_supply': coin_max_supply
    }
#Now each coin_metadata is like a row in a table.
#We append this to our list of rows:
    metadata_list.append(coin_metadata)

In [15]:
metadata_list

[{'coin_id': 'bitcoin',
  'symbol': 'btc',
  'name': 'Bitcoin',
  'image_url': 'https://coin-images.coingecko.com/coins/images/1/large/bitcoin.png?1696501400',
  'market_cap_rank': 1,
  'max_supply': 21000000.0},
 {'coin_id': 'ethereum',
  'symbol': 'eth',
  'name': 'Ethereum',
  'image_url': 'https://coin-images.coingecko.com/coins/images/279/large/ethereum.png?1696501628',
  'market_cap_rank': 2,
  'max_supply': None},
 {'coin_id': 'tether',
  'symbol': 'usdt',
  'name': 'Tether',
  'image_url': 'https://coin-images.coingecko.com/coins/images/325/large/Tether.png?1696501661',
  'market_cap_rank': 3,
  'max_supply': None},
 {'coin_id': 'ripple',
  'symbol': 'xrp',
  'name': 'XRP',
  'image_url': 'https://coin-images.coingecko.com/coins/images/44/large/xrp-symbol-white-128.png?1696501442',
  'market_cap_rank': 4,
  'max_supply': 100000000000.0},
 {'coin_id': 'binancecoin',
  'symbol': 'bnb',
  'name': 'BNB',
  'image_url': 'https://coin-images.coingecko.com/coins/images/825/large/bnb-i

🧱 Step 3: Build a Clean List of Market DAta

We only want certain descriptive fields about each coin (not the changing prices or volumes). So we loop through the list and extract the metadata fields we care about:

In [33]:
market_data_list = []

for coin in raw_data:
    coin_id = coin['id']
    current_price = coin['current_price']
    market_cap = coin['market_cap']
    total_volume = coin['total_volume']
    high_24h = coin['high_24h']
    low_24h = coin['low_24h']
    price_change_24h = coin['price_change_24h']
    price_change_pct_24h = coin['price_change_percentage_24h']
    market_cap_change_24h = coin['market_cap_change_24h']
    market_cap_change_pct_24h = coin['market_cap_change_percentage_24h']

    coin_market_data = {
        'coin_id': coin_id,
        'current_price': current_price,
        'market_cap': market_cap,
        'total_volume': total_volume,
        'high_24h': high_24h,
        'low_24h': low_24h,
        'price_change_24h': price_change_24h,
        'price_change_pct_24h': price_change_pct_24h,
        'market_cap_change_24h': market_cap_change_24h,
        'market_cap_change_pct_24h': market_cap_change_pct_24h
    }

    market_data_list.append(coin_market_data)


In [43]:
#market_data_list

📊 Step 4: Convert to DataFrame

Now that we have a list of dictionaries (like a table), we use pandas to convert it into a real DataFrame:

In [35]:
import pandas as pd
df_metadata = pd.DataFrame(metadata_list)

In [38]:
# Display DataFrame
df_metadata.head()

,coin_id,symbol,name,image_url,market_cap_rank,max_supply
0,bitcoin,btc,Bitcoin,https://coin-images.coingecko.com/coins/images...,1,2.100000e+07
1,ethereum,eth,Ethereum,https://coin-images.coingecko.com/coins/images...,2,NaN
2,tether,usdt,Tether,https://coin-images.coingecko.com/coins/images...,3,NaN
3,ripple,xrp,XRP,https://coin-images.coingecko.com/coins/images...,4,1.000000e+11
4,binancecoin,bnb,BNB,https://coin-images.coingecko.com/coins/images...,5,2.000000e+08


In [39]:
#get a summary of data
df_metadata.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   coin_id          50 non-null     object 
 1   symbol           50 non-null     object 
 2   name             50 non-null     object 
 3   image_url        50 non-null     object 
 4   market_cap_rank  50 non-null     int64  
 5   max_supply       23 non-null     float64
dtypes: float64(1), int64(1), object(4)
memory usage: 2.5+ KB


In [40]:
df_metadata = df_metadata.drop_duplicates(subset=['coin_id'])

In [42]:
df_marketdata = pd.DataFrame(market_data_list)

In [45]:
df_marketdata.head()

,coin_id,current_price,market_cap,total_volume,high_24h,low_24h,price_change_24h,price_change_pct_24h,market_cap_change_24h,market_cap_change_pct_24h
0,bitcoin,107249.00,2132660458082,9.563125e+09,107530.00,106905.00,145.800000,0.13613,3.213395e+09,0.15090
1,ethereum,2433.09,293658499059,5.103819e+09,2445.34,2410.91,9.310000,0.38403,1.126715e+09,0.38516
2,tether,1.00,157523409654,1.308218e+10,1.00,1.00,-0.000096,-0.00960,-6.772592e+06,-0.00430
3,ripple,2.18,128866085651,2.088537e+09,2.21,2.11,0.076984,3.65312,4.624205e+09,3.72194
4,binancecoin,647.89,94519624560,3.087252e+08,648.02,644.56,1.450000,0.22357,2.199967e+08,0.23330


In [ ]:
df_marketdata.info()


In [46]:
df_marketdata = df_marketdata.drop_duplicates(subset=['coin_id'])